## 1. Общий setup

Заполните `REPO_URL`, при необходимости поменяйте `BRANCH`, затем выполните блок. CSV-файлы должны лежать в корне репозитория.

In [ ]:
# 1. Общий setup
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Timur713/HSE_RAG_Hackaton"  # example: "https://github.com/<user>/<repo>.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/legal_hse_4")
RUN_OPTIONAL_DENSE = False  # True installs sentence-transformers and enables dense configs

if not REPO_URL:
    raise ValueError("Заполните REPO_URL перед запуском setup-блока.")

if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR)], check=True)
if RUN_OPTIONAL_DENSE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-optional.txt")], check=True)

os.chdir(PROJECT_DIR)
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from legal_hse.config import PathConfig

paths = PathConfig.from_root(PROJECT_DIR)
paths.ensure_dirs()
paths.require_input_files()
print(f"Project ready: {PROJECT_DIR}")

## 2. Все эксперименты

`holdout` быстрее для рабочих итераций. `cv` используйте для проверки лучших конфигураций.

In [ ]:
# 2. Все эксперименты
from legal_hse.experiments import run_suite, select_best_experiment

MODE = "holdout"  # "holdout", "cv" or "train"
EXPERIMENT_NAMES = None  # None runs default fast suite; or list names, e.g. ["bm25_doc", "rrf_bm25_doc_chunk"]
SEED = 42
N_SPLITS = 5

summary = run_suite(
    data_dir=PROJECT_DIR,
    output_dir=PROJECT_DIR / "reports",
    experiment_names=EXPERIMENT_NAMES,
    mode=MODE,
    include_optional=RUN_OPTIONAL_DENSE,
    seed=SEED,
    n_splits=N_SPLITS,
    eval_depth=50,
)

cols = ["split", "experiment", "status", "recall@5", "recall@10", "recall@20", "recall@50", "duration_sec"]
display(summary[[c for c in cols if c in summary.columns]].sort_values(["status", "recall@5"], ascending=[True, False]))
BEST_EXPERIMENT = select_best_experiment(summary)
print("BEST_EXPERIMENT =", BEST_EXPERIMENT)

## 3. Загрузка результатов на GitHub

Введите `GITHUB_USERNAME`, `GIT_EMAIL`, `SSH_PRIVATE_KEY_B64`. Ключ должен иметь write-доступ к репозиторию.

In [ ]:
# 3. Загрузка результатов [метрик] на GitHub
import base64
import getpass
import os
import subprocess
from pathlib import Path

GITHUB_USERNAME = input("GITHUB_USERNAME: ").strip()
GIT_EMAIL = input("GIT_EMAIL: ").strip()
SSH_PRIVATE_KEY_B64 = getpass.getpass("SSH_PRIVATE_KEY_B64: ").strip()
SSH_REMOTE_URL = ""  # optional: git@github.com:<user>/<repo>.git

ssh_dir = Path.home() / ".ssh"
ssh_dir.mkdir(mode=0o700, exist_ok=True)
key_path = ssh_dir / "id_ed25519"
key_path.write_bytes(base64.b64decode(SSH_PRIVATE_KEY_B64))
key_path.chmod(0o600)
subprocess.run(["ssh-keyscan", "github.com"], stdout=(ssh_dir / "known_hosts").open("a"), check=True)

subprocess.run(["git", "config", "user.name", GITHUB_USERNAME], check=True)
subprocess.run(["git", "config", "user.email", GIT_EMAIL], check=True)

if not SSH_REMOTE_URL and REPO_URL.startswith("https://github.com/"):
    repo_path = REPO_URL.split("https://github.com/", 1)[1]
    repo_path = repo_path[:-4] if repo_path.endswith(".git") else repo_path
    SSH_REMOTE_URL = f"git@github.com:{repo_path}.git"
if SSH_REMOTE_URL:
    subprocess.run(["git", "remote", "set-url", "origin", SSH_REMOTE_URL], check=True)

paths_to_add = ["reports/metrics", "reports/summary_latest.csv"]
paths_to_add += [str(path) for path in Path("reports").glob("summary_*.csv")]
subprocess.run(["git", "add", *paths_to_add], check=True)
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, check=True).stdout.strip()
if status:
    subprocess.run(["git", "commit", "-m", "Add retrieval experiment metrics"], check=True)
    subprocess.run(["git", "push", "origin", f"HEAD:{BRANCH}"], check=True)
    print("Metrics pushed to GitHub.")
else:
    print("No metric changes to push.")

## 4. Формирование submission файла

По умолчанию используется лучший experiment name из блока 2. Его можно заменить вручную.

In [ ]:
# 4. Формирование submission файла
import pandas as pd
from legal_hse.experiments import create_submission

FINAL_EXPERIMENT = globals().get("BEST_EXPERIMENT", "rrf_bm25_doc_chunk")
SUBMISSION_PATH = create_submission(
    data_dir=PROJECT_DIR,
    experiment_name=FINAL_EXPERIMENT,
    output_path=PROJECT_DIR / "submissions" / f"submission_{FINAL_EXPERIMENT}.csv",
    include_optional=RUN_OPTIONAL_DENSE,
    top_k=5,
)
print("Submission:", SUBMISSION_PATH)
display(pd.read_csv(SUBMISSION_PATH).head(15))